In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("data/raw")
PROCESSED = Path("data/processed")

nav = pd.read_csv(RAW/"02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [2]:
# date conversion
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

# sort
nav = nav.sort_values(
    ["amfi_code","date"]
)

# remove duplicates
nav = nav.drop_duplicates()

# forward fill NAV holidays/weekends
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()


# validate NAV
nav = nav[nav["nav"] > 0]


nav.to_csv(
    PROCESSED/"nav_history_cleaned.csv",
    index=False
)


nav.shape

(46000, 3)

In [5]:
txn = pd.read_csv(
    "data/raw/08_investor_transactions.csv"
)

print(txn.columns)


# standardize transaction type
txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.upper()
)

txn["transaction_type"] = txn["transaction_type"].replace({
    "LUMPSUM":"Lumpsum",
    "SIP":"SIP",
    "REDEMPTION":"Redemption"
})


# date conversion
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)


# find amount column
amount_col = [
    c for c in txn.columns 
    if "amount" in c.lower()
][0]


# validate amount
txn = txn[txn[amount_col] > 0]


# kyc validation
valid_kyc = [
    "Verified",
    "Pending",
    "Rejected"
]

txn = txn[
    txn["kyc_status"].isin(valid_kyc)
]


txn.to_csv(
    "data/processed/transactions_cleaned.csv",
    index=False
)


txn.head()

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [2]:
import pandas as pd

perf = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)

print(perf.columns.tolist())

perf.head()

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [3]:
# Convert numeric columns safely

for col in perf.columns:
    if any(x in col.lower() for x in ["return", "expense", "ratio"]):
        perf[col] = pd.to_numeric(
            perf[col],
            errors="coerce"
        )


# check missing values
print("Missing values:")
print(perf.isnull().sum())


# validate expense ratio range
expense_col = [
    c for c in perf.columns 
    if "expense" in c.lower()
][0]


perf = perf[
    (perf[expense_col] >= 0.1) &
    (perf[expense_col] <= 2.5)
]


# save cleaned file

perf.to_csv(
    "data/processed/performance_cleaned.csv",
    index=False
)


print("Saved performance_cleaned.csv")
print(perf.shape)

Missing values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64
Saved performance_cleaned.csv
(40, 19)


In [7]:
from pathlib import Path

Path("database").mkdir(exist_ok=True)

print("database folder created")

database folder created


In [8]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///database/bluestock_mf.db"
)

print("SQLite connected")

SQLite connected


In [11]:
import pandas as pd

# reload cleaned datasets

nav = pd.read_csv(
    "data/processed/nav_history_cleaned.csv"
)

txn = pd.read_csv(
    "data/processed/transactions_cleaned.csv"
)

perf = pd.read_csv(
    "data/processed/performance_cleaned.csv"
)

fund = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

print("All datasets loaded")
print(nav.shape)
print(txn.shape)
print(perf.shape)
print(fund.shape)

All datasets loaded
(46000, 3)
(32778, 13)
(40, 19)
(40, 15)


In [12]:
fund.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

txn.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

perf.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("✅ SQLite Database Loaded")

✅ SQLite Database Loaded


In [13]:
from sqlalchemy import inspect

inspector = inspect(engine)

print(inspector.get_table_names())

['dim_fund', 'fact_nav', 'fact_performance', 'fact_transactions']


In [14]:
from sqlalchemy import text

tables = [
    "dim_fund",
    "fact_nav",
    "fact_transactions",
    "fact_performance"
]

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        print(table, ":", count)

dim_fund : 40
fact_nav : 46000
fact_transactions : 32778
fact_performance : 40
